In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("dataset.csv")

In [3]:
queries = df['query'].values
labels = df['label'].values

In [4]:
from sklearn.model_selection import train_test_split

In [5]:
X_train, X_temp, y_train, y_temp = train_test_split(
    queries,
    labels,
    test_size=0.3,
    stratify=labels,
    random_state=42
)

In [6]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

In [8]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=10000,
    lowercase=True,
    stop_words="english",
    token_pattern=r"(?u)\b\w\w+\b",
    norm="l2",
    use_idf=True,
    smooth_idf=True,
    sublinear_tf=True,
)

In [9]:
pipeline = Pipeline([
    ("tfidf", vectorizer),
    ("clf", LogisticRegression(
        max_iter=1000,
        class_weight={"search": 1.25, "llm": 1.0}
    ))
])

In [10]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=10000, ngram_range=(1, 2),
                                 stop_words='english', sublinear_tf=True)),
                ('clf',
                 LogisticRegression(class_weight={'llm': 1.0, 'search': 1.25},
                                    max_iter=1000))])

In [18]:
y_val_pred = pipeline.predict(X_val)

In [16]:
y_test_pred = pipeline.predict(X_test)

In [13]:
probs = pipeline.predict_proba(X_test)
max_probs = probs.max(axis=1)

In [15]:
import pickle

with open("query_router.pkl", "wb") as f:
    pickle.dump(pipeline, f)

print("Model saved successfully.")

Model saved successfully.
